In [2]:
import sqlite3
import datetime
from datetime import timedelta
import pandas as pd

In [6]:
def read_database(db_path, table_name='itin'):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    cursor.execute(f'SELECT * FROM {table_name}')
    rows = cursor.fetchall()
    
    data = pd.DataFrame(rows, columns=[col[0] for col in cursor.description])

    conn.close()
    return data

def read_parquet(parquet_path):
    return pd.read_parquet(parquet_path)

def itineraries_list(offsets):
    """
    Returns a list of itineraries from the database for the target dates

    """
    db_path = "flights.db"  # Replace with your database path
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    today = datetime.date.today()
    target_dates = [(today + datetime.timedelta(days=o)).isoformat() for o in offsets]

    # Now use these dates in your SQL query:
    placeholders = ','.join('?' for _ in target_dates)
    query = f"""
        SELECT itinerary_id, departure_date
        FROM itin
        WHERE departure_date IN ({placeholders})
    """
    cursor.execute(query, target_dates)
    rows = cursor.fetchall()

    itineraries = pd.DataFrame(rows, columns=[col[0] for col in cursor.description])
    conn.close()
    return itineraries


In [18]:
offsets = (
        list(range(1, 8, 1)) +        # daily first week
        list(range(8, 31, 2)) +       # every 2 days next 3 weeks
        list(range(31, 91, 4))        # every 4 days up to 90 days
)
existing_itineraries = itineraries_list(offsets)
# Read the parquet file
prices = read_parquet("prices_latest.parquet")


In [59]:
# Checked missed itineraries
missing_itineraries1 = existing_itineraries[~existing_itineraries['itinerary_id'].isin(prices['itinerary_id'])]

print("Missing itineraries 1st check:")
print(missing_itineraries1.shape[0])

# Check missed itineraries only excluding flight code
# Extract the first part before 'T' for both DataFrames
# For a DataFrame column 'itinerary_id':
missing_itineraries1[['flight_number', 'date', 'time', 'origin', 'destination']] = (
    missing_itineraries1['itinerary_id']
    .str.extract(r'([A-Z0-9]+)-(\d{4}-\d{2}-\d{2})T(\d{2}:\d{2}:\d{2})-([A-Z]+)-([A-Z]+)')
)
prices[['flight_number', 'date', 'time', 'origin', 'destination']] = (
    prices['itinerary_id']
    .str.extract(r'([A-Z0-9]+)-(\d{4}-\d{2}-\d{2})T(\d{2}:\d{2}:\d{2})-([A-Z]+)-([A-Z]+)')
)

cols_to_check1 = ['flight_number', 'date', 'origin', 'destination']
cols_to_check2 = ['date', 'time', 'origin', 'destination']
merged = pd.merge(
    missing_itineraries1,
    prices[cols_to_check1].drop_duplicates(),
    on=cols_to_check1,
    how='left',
    indicator=True
)
missing_by_first_part = merged[merged['_merge'] == 'left_only'][missing_itineraries1.columns]
print("Missing itineraries by first part:")
print(missing_by_first_part.shape[0])
        

Missing itineraries 1st check:
643


C:\Users\pinyc\AppData\Local\Temp\ipykernel_24840\2892256158.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  missing_itineraries1[['flight_number', 'date', 'time', 'origin', 'destination']] = (
C:\Users\pinyc\AppData\Local\Temp\ipykernel_24840\2892256158.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  missing_itineraries1[['flight_number', 'date', 'time', 'origin', 'destination']] = (
C:\Users\pinyc\AppData\Local\Temp\ipykernel_24840\2892256158.py:10: SettingWithCopyWarning: 
A value is trying to 

Missing itineraries by first part:
465


In [46]:
missing_by_first_part.head(20)

,itinerary_id,departure_date,first_part
115,AA100-2025-06-26T18:10:00-JFK-LHR,2025-06-26,AA100-2025-06-26
116,AA100-2025-06-27T18:10:00-JFK-LHR,2025-06-27,AA100-2025-06-27
141,AA104-2025-06-26T23:00:00-JFK-LHR,2025-06-26,AA104-2025-06-26
142,AA104-2025-06-27T23:00:00-JFK-LHR,2025-06-27,AA104-2025-06-27
175,AA142-2025-06-27T09:45:00-JFK-LHR,2025-06-27,AA142-2025-06-27
177,AA142-2025-06-29T09:45:00-JFK-LHR,2025-06-29,AA142-2025-06-29
222,AA161-2025-06-27T11:05:00-ATH-ORD,2025-06-27,AA161-2025-06-27
266,AA333-2025-06-26T15:55:00-ATH-JFK,2025-06-26,AA333-2025-06-26
267,AA333-2025-06-27T15:55:00-ATH-JFK,2025-06-27,AA333-2025-06-27
269,AA333-2025-06-29T15:55:00-ATH-JFK,2025-06-29,AA333-2025-06-29
